# 09 — RAG Limitations & Simple Evaluation

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Recognise the 8 most common RAG failure modes.
2. Reproduce 2-3 of them with our own data.
3. Build a tiny evaluation harness with expected keywords + expected sources.
4. Apply a *Safe-Use Checklist* before relying on RAG for client work.


## The 8 failure modes

1. **Hallucination** — model invents facts/citations.
2. **Incomplete source corpus** — the answer is *not* in our docs.
3. **Poor chunking** — a key fact is split across two chunks.
4. **OCR / scanned PDFs** — extraction returns empty text.
5. **Tables** — `pypdf` flattens tables into garbled rows.
6. **Wrong retrieval** — semantically close but factually wrong chunk wins.
7. **Missing metadata** — can't filter or attribute correctly.
8. **Outdated regulation** — model's training is frozen; laws change.
9. **Confidentiality** — sending client data to a public API.
10. **Overreliance** — trusting AI output without verification.


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


In [ ]:
from src.rag_utils import build_store_from_folder, rag_answer
store = build_store_from_folder('data/generated/pdf')
print('Store size:', len(store))

## 9.1 — Reproduce: hallucination from an out-of-corpus question

In [ ]:
print(rag_answer('What is the current corporate tax rate in Bhutan?', store, k=3))
# Expected: the model should answer "I could not find this in the provided documents."

## 9.2 — Reproduce: wrong retrieval

In [ ]:
# 'approval matrix' is a phrase in the Internal Control Policy.
# But the term may also be (weakly) similar to chunks in the procurement and management letter.
answer, hits = rag_answer('What is the approval matrix?', store, k=1, return_hits=True)
print(answer)
for h in hits:
    print('  retrieved:', h['metadata'].get('source'), 'p.', h['metadata'].get('page'))

Try the same with `k=1` versus `k=4`. With `k=1` retrieval misses are *fatal*; with `k=4` the right chunk usually survives. This is the classic recall-vs-precision trade-off.

## 9.3 — Reproduce: poor chunking

In [ ]:
from src.rag_utils import VectorStore, chunk_documents
from src.document_loaders import load_pdfs_in_folder

docs = load_pdfs_in_folder('data/generated/pdf')
# Deliberately tiny chunks → important threshold numbers may be cut in half
tiny_chunks = chunk_documents(docs, chunk_size=120, overlap=0)
bad_store = VectorStore(); bad_store.add(tiny_chunks)
print('Tiny-chunk answer:')
print(rag_answer('What is the DSCR covenant on the term loan?', bad_store, k=3))

## 9.4 — A small evaluation set

In [ ]:
from src.evaluation_utils import EvalCase, evaluate
import pandas as pd

cases = [
    EvalCase(
        question='What are the five significant audit risks?',
        expected_keywords=['revenue', 'inventory', 'related party', 'covenant', 'going concern'],
        expected_source_contains='audit_planning',
    ),
    EvalCase(
        question='What is the DSCR covenant?',
        expected_keywords=['1.25', 'dscr', 'debt-service'],
        expected_source_contains='loan_agreement',
    ),
    EvalCase(
        question='Which approval is required for purchases above NPR 5 lakh?',
        expected_keywords=['cfo', 'ceo', 'board', '5,00,000'],
        expected_source_contains='internal_control',
    ),
    EvalCase(
        question='List two related-party entities and their nature.',
        expected_keywords=['annapurna', 'himal family', 'common director', 'family'],
        expected_source_contains='annual_report',
    ),
    EvalCase(
        question='When does the financial year end?',
        expected_keywords=['2082', 'ashad', '03-31'],
        expected_source_contains=None,
    ),
]
def fn(q):
    return rag_answer(q, store, k=4, return_hits=True)
rows = evaluate(cases, fn)
display(pd.DataFrame(rows))

## 9.5 — The Safe-Use Checklist

In [ ]:
from src.evaluation_utils import SAFE_USE_CHECKLIST
print(SAFE_USE_CHECKLIST)

## Exercise

1. Add three of your own eval cases above. Aim for at least one *unanswerable* case.
2. Pick the worst-performing case and improve it by **only** changing the chunk size — no LLM swap.
3. Discuss with a partner: which failure mode is the most dangerous for your day-to-day work?


## ⚠️ Professional caution

RAG is *plumbing*, not *judgement*. It improves grounding but does not guarantee correctness. A score of 100% on five eval cases does not justify using the system unsupervised on a client engagement.